# Task 3: Information Processing + Storage

This notebook processes the raw collected documents and stores them in ChromaDB.

Steps:
1. Load raw documents from Task 1
2. Clean the data
3. Remove duplicates
4. Extract relevant information (keep only Lufthansa-related)
5. Generate embeddings (multilingual for English + German)
6. Index into ChromaDB for retrieval

This completes both Task 3 (processing) and Task 2 (knowledge repository).

In [23]:
import json
import os
import re
from datetime import datetime
import chromadb
from sentence_transformers import SentenceTransformer

## Step 1: Load Raw Documents

We load the 173 documents collected in Notebook 01.

In [24]:
# Load the raw collected documents from Task 1
with open("../data/collected_documents.json", "r", encoding="utf-8") as f:
    documents = json.load(f)

print(f"Total raw documents loaded: {len(documents)}")
print(f"\nSample document:")
print(f"  Title    : {documents[0]['title']}")
print(f"  Summary  : {documents[0]['summary'][:80]}...")
print(f"  Category : {documents[0]['category']}")
print(f"  Source   : {documents[0]['source']}")

Total raw documents loaded: 476

Sample document:
  Title    : Newsroom Deutsche Lufthansa AG
  Summary  : Lufthansa Group at ILA Berlin 2026: Anniversary Aircraft and Lufthansa Technik D...
  Category : company
  Source   : DuckDuckGo Search


## Step 2: Clean the Data

We clean each document by:
- Removing extra spaces and newlines
- Removing HTML tags and special characters that add noise
- Keeping the text readable for embeddings

In [25]:
# Function to clean a piece of text
def clean_text(text):

    # Remove HTML tags like <p> or <br>
    text = re.sub(r"<.*?>", " ", text)

    # Remove extra spaces, tabs, and newlines
    text = re.sub(r"\s+", " ", text)

    # Remove strange special characters but keep normal punctuation
    text = re.sub(r"[^\w\s.,!?€%-]", " ", text)

    # Remove extra spaces again and trim
    text = text.strip()

    return text


# Clean every document's title and summary
for doc in documents:
    doc["title"]   = clean_text(doc["title"])
    doc["summary"] = clean_text(doc["summary"])

print("All documents cleaned")
print(f"\nSample cleaned document:")
print(f"  Title   : {documents[0]['title']}")
print(f"  Summary : {documents[0]['summary'][:100]}...")

All documents cleaned

Sample cleaned document:
  Title   : Newsroom Deutsche Lufthansa AG
  Summary : Lufthansa Group at ILA Berlin 2026  Anniversary Aircraft and Lufthansa Technik Defense ... Press rel...


## Step 3: Remove Duplicates

Some documents may have the same title or very similar content.

In [26]:
# Remove duplicate documents based on title
seen_titles = set()
unique_documents = []

for doc in documents:
    title = doc["title"].lower().strip()

    if title not in seen_titles:
        seen_titles.add(title)
        unique_documents.append(doc)

print(f"Before removing duplicates : {len(documents)}")
print(f"After removing duplicates  : {len(unique_documents)}")
print(f"Duplicates removed         : {len(documents) - len(unique_documents)}")

Before removing duplicates : 476
After removing duplicates  : 458
Duplicates removed         : 18


In [27]:
# Also remove duplicates based on URL (same article, different title)
seen_urls = set()
final_unique = []

for doc in unique_documents:
    url = doc["url"].strip()

    # Keep document if URL is new, or if it has no URL
    if url == "" or url not in seen_urls:
        if url != "":
            seen_urls.add(url)
        final_unique.append(doc)

unique_documents = final_unique

print(f"After URL deduplication : {len(unique_documents)}")

After URL deduplication : 457


## Step 4: Extract Relevant Information

The RSS feeds gave us general aviation news — some articles may not be 
about Lufthansa at all (e.g. Emirates, Boeing news).

We keep only documents that actually mention Lufthansa or its subsidiaries.
This ensures our knowledge base is focused on the company we are analyzing.

In [28]:
# Keywords that show a document is relevant to Lufthansa Group
lufthansa_keywords = [
    "lufthansa", "eurowings", "swiss", "austrian airlines",
    "brussels airlines", "miles & more", "miles and more",
    "lufthansa cargo", "lufthansa technik", "star alliance",
    "carsten spohr", "lufthansa hanger"
]

relevant_documents = []

for doc in unique_documents:

    # Combine title and summary, make lowercase for checking
    text = (doc["title"] + " " + doc["summary"]).lower()

    # Check if any Lufthansa keyword appears
    is_relevant = any(keyword in text for keyword in lufthansa_keywords)

    if is_relevant:
        relevant_documents.append(doc)

print(f"Before relevance filter : {len(unique_documents)}")
print(f"After relevance filter  : {len(relevant_documents)}")
print(f"Removed (not relevant)  : {len(unique_documents) - len(relevant_documents)}")

Before relevance filter : 457
After relevance filter  : 429
Removed (not relevant)  : 28


In [29]:
# Check category distribution after filtering
category_count = {}
source_count = {}

for doc in relevant_documents:
    cat = doc["category"]
    src = doc["source"]
    category_count[cat] = category_count.get(cat, 0) + 1
    source_count[src] = source_count.get(src, 0) + 1

print("Documents by category:")
for cat, count in category_count.items():
    print(f"  {cat}: {count}")

print("\nDocuments by source:")
for src, count in source_count.items():
    print(f"  {src}: {count}")

print(f"\nTotal relevant documents: {len(relevant_documents)}")

Documents by category:
  company: 47
  news: 280
  market: 36
  community: 28
  research: 38

Documents by source:
  DuckDuckGo Search: 167
  Wikipedia (EN): 8
  Wikipedia (DE): 1
  Google News: 88
  Google News Strategy: 72
  Google News Finance: 93

Total relevant documents: 429


## Step 5: Generate Embeddings

We convert each document's text into embeddings (numerical vectors).
We use a multilingual model so the system understands both English and German.

Model: paraphrase-multilingual-MiniLM-L12-v2
- Supports 50+ languages including English and German
- Lightweight and fast
- Free to use

In [34]:
# Load the multilingual embedding model
# This is important because we have both English and German documents
from dotenv import load_dotenv
import os

load_dotenv("../.env")

model = SentenceTransformer(
    "paraphrase-multilingual-MiniLM-L12-v2",
    token=os.getenv("HF_TOKEN")
)

print("Embedding model loaded")
print("Supports English, German, and 50+ other languages")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 6353.99it/s]


Embedding model loaded
Supports English, German, and 50+ other languages


## Step 6: Store Documents in ChromaDB

We prepare documents and store them with their embeddings in ChromaDB.
ChromaDB becomes our knowledge repository — ready for semantic search.

In [ ]:
# Setup ChromaDB - persistent storage
chroma_client = chromadb.PersistentClient(path="../storage/chromadb")

# Delete old collection if exists and create fresh one
try:
    chroma_client.delete_collection("lufthansa_intelligence")
    print("Old collection deleted")
except:
    pass

collection = chroma_client.get_or_create_collection(
    name="lufthansa_intelligence",
    metadata={"hnsw:space": "cosine"}
)

# Prepare documents for ChromaDB
ids       = []
texts     = []
metadatas = []

for i, doc in enumerate(relevant_documents):

    # Combine title and summary as the main searchable text
    text = f"{doc['title']}. {doc['summary']}"

    ids.append(f"doc_{i}")
    texts.append(text)
    metadatas.append({
        "title"    : doc["title"],
        "category" : doc["category"],
        "source"   : doc["source"],
        "date"     : doc["date"],
        "url"      : doc["url"]
    })

print(f"Documents prepared : {len(ids)}")

# Generate embeddings using our multilingual model
print("Generating embeddings...")
embeddings = model.encode(texts, show_progress_bar=True)

# Store everything in ChromaDB
collection.add(
    ids        = ids,
    documents  = texts,
    embeddings = embeddings.tolist(),
    metadatas  = metadatas
)

print(f"\nDocuments stored in ChromaDB : {collection.count()}")
print("Knowledge repository is ready")

Old collection deleted
Documents prepared : 429
Generating embeddings...


Batches: 100%|██████████| 14/14 [00:03<00:00,  4.48it/s]



Documents stored in ChromaDB : 429
Knowledge repository is ready


In [35]:
# Test semantic search - English and German
query_en = "What are Lufthansa's biggest risks?"
query_de = "Was sind die größten Risiken für Lufthansa?"

for query in [query_en, query_de]:
    query_embedding = model.encode([query]).tolist()

    results = collection.query(
        query_embeddings = query_embedding,
        n_results        = 3
    )

    print(f"Query: {query}")
    for i in range(3):
        print(f"  {i+1}. {results['metadatas'][0][i]['title']}")
    print()

Query: What are Lufthansa's biggest risks?
  1. Lufthansa  Hoffnung auf Fracht und US-Geschäft - WELT
  2. Geschäftsbericht 2024
  3. Streit um Drehkreuze  Emirates und Co. setzen Lufthansa zu

Query: Was sind die größten Risiken für Lufthansa?
  1. Lufthansa  Hoffnung auf Fracht und US-Geschäft - WELT
  2. Geschäftsbericht 2024
  3. Erstes Halbjahr erfolgreich  Nach Gewinnsprung  Lufthansa auf Kurs...

